# Z integrity metric

ovrlpy gives each 1 µm pixel a vertical signal integrity (VSI). Low VSI (< 0.7) means the transcripts above and below differ, so the signal mixes cells stacked in z.

For each method and sample, `compute_z_integrity` pools all pixels inside the method's post-QC cells, keeps pixels with signal > 2 and reports the % with low VSI (`low_vsi_pct`, lower is better). Pooling makes the score independent of how a region is split into cells, so oversegmentation is not penalised. Coverage (% of signal pixels inside cells) is reported beside it, because methods that cover less tissue can avoid or hit low-VSI regions.

Results: `metrics/<cohort>/z_integrity/z_integrity_all.csv` (and `_vascular_celltypes.csv` for the vascular subset), one row per sample and threshold pair (signal 1/2/3, VSI 0.6/0.7/0.8), plus `all_samples` rows with the mean over samples. Needs the VSI maps from `scripts/seg_postprocessing/ovrlpy_vsi_map.py`.

## Compute

In [ ]:
from cellseg_benchmark.metrics import (
    compute_metric_for_all_methods,
    compute_z_integrity,
)

cohorts = ["aging", "ABCAtlas", "VizgenMouseBrain"]
for cohort in cohorts:
    compute_metric_for_all_methods(compute_z_integrity, cohort=cohort, results_name="z_integrity/z_integrity_all.csv")
compute_metric_for_all_methods(compute_z_integrity, cohort="aging", results_name="z_integrity/z_integrity_vascular_celltypes.csv",
                               adata_name="adata_vascular_subset")

## Load results

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from cellseg_benchmark import BASE_PATH
from cellseg_benchmark._constants import method_colors
from cellseg_benchmark.metrics import clean_method_name

base = Path(BASE_PATH)
plot_path = base / "metrics" / "z_integrity_plots"
plot_path.mkdir(parents=True, exist_ok=True)

df = pd.concat(pd.read_csv(base / "metrics" / c / "z_integrity" / "z_integrity_all.csv", index_col=0).assign(cohort=c) for c in cohorts)
df = df[(df["sample"] != "all_samples") & ~df["method"].eq("Negative_Control_Visium")]
main = df.query("signal_threshold == 2 and vsi_threshold == 0.7")
means = main.groupby(["cohort", "method"])[["low_vsi_pct", "coverage_pct", "mean_vsi"]].mean()
means["rank"] = means.groupby(level=0)["low_vsi_pct"].rank()
means["low_vsi_pct"].unstack(0).round(2).assign(mean_rank=means["rank"].unstack(0).mean(1)).sort_values("mean_rank")

## 1. Low-VSI share per cohort

Dots are samples, ticks are medians. Coverage on the right.

In [ ]:
n = main["method"].nunique()
fig, axs = plt.subplots(len(cohorts), 2, figsize=(9, 0.25 * n * len(cohorts) + 1), gridspec_kw={"width_ratios": [3, 1]})
for i, cohort in enumerate(cohorts):
    d = main[main["cohort"] == cohort].assign(label=lambda x: x["method"].map(clean_method_name))
    order = d.groupby("label")["low_vsi_pct"].median().sort_values().index
    pal = dict(zip(d["label"], d["method"].map(lambda m: method_colors.get(m, "0.5"))))
    for ax, col, xlabel in [(axs[i, 0], "low_vsi_pct", "Low-VSI signal in cells (%)"), (axs[i, 1], "coverage_pct", "Coverage (%)")]:
        sns.stripplot(d, x=col, y="label", hue="label", order=order, palette=pal, size=3, jitter=0.2, legend=False, ax=ax)
        sns.boxplot(d, x=col, y="label", order=order, showbox=False, showcaps=False, whiskerprops={"visible": False},
                    showfliers=False, medianprops={"color": "k", "lw": 1.5}, ax=ax)
        ax.set(xlabel=xlabel, ylabel="", title=cohort if col == "low_vsi_pct" else "")
        ax.grid(axis="x", alpha=0.3)
        if col == "coverage_pct":
            ax.set(xlim=(0, 100), yticklabels=[])
plt.tight_layout()
plt.savefig(plot_path / "z_integrity_per_cohort.png", dpi=200, bbox_inches="tight")
plt.show()

## 2. Low-VSI share vs coverage

Methods covering almost all tissue sit at the tissue average. All spread between methods comes from methods with lower coverage.

In [ ]:
fig, axs = plt.subplots(1, len(cohorts), figsize=(5 * len(cohorts), 4.5))
for ax, cohort in zip(axs, cohorts):
    m = means.loc[cohort].reset_index()
    ax.scatter(m["coverage_pct"], m["low_vsi_pct"], c=m["method"].map(lambda x: method_colors.get(x, "0.5")), s=30, edgecolor="k", lw=0.3)
    for _, r in m.nlargest(3, "low_vsi_pct").iterrows():
        ax.annotate(clean_method_name(r["method"]), (r["coverage_pct"], r["low_vsi_pct"]), fontsize=7)
    rho = m[["coverage_pct", "low_vsi_pct"]].corr("spearman").iloc[0, 1]
    ax.set(xlabel="Coverage (%)", ylabel="Low-VSI signal in cells (%)", title=f"{cohort} (rho {rho:.2f})")
plt.tight_layout()
plt.savefig(plot_path / "z_integrity_vs_coverage.png", dpi=200, bbox_inches="tight")
plt.show()

## 3. Sensitivity

Spearman correlation of the method order (mean over samples) at each threshold pair with the default (signal 2, VSI 0.7), and how well samples within a cohort agree (mean pairwise Spearman between samples).

In [ ]:
order_at = df.groupby(["cohort", "signal_threshold", "vsi_threshold", "method"])["low_vsi_pct"].mean().unstack("method")
default = order_at.xs((2, 0.7), level=("signal_threshold", "vsi_threshold"))
sens = pd.DataFrame({c: order_at.loc[c].T.corrwith(default.loc[c], method="spearman") for c in cohorts}).round(2)

def sample_agreement(d):
    r = d.pivot(index="method", columns="sample", values="low_vsi_pct").corr("spearman").to_numpy()
    return r[np.triu_indices_from(r, 1)].mean()

print(main.groupby("cohort").apply(sample_agreement).round(2).to_string())
sens

## 4. Do low-VSI cells carry mixed signal?

Validity check: cells with at least half of their signal pixels at low VSI should express more markers of other cell types. For a few methods and samples, marker F1 and the marker false positive rate (cells of other types expressing the marker) are compared between low-VSI cells and the rest, within quartiles of transcript counts so that depth does not drive the difference. Positive FPR differences and negative F1 differences support the metric.

In [ ]:
from cellseg_benchmark.metrics import (
    compute_marker_F1_score,
    low_vsi_per_cell,
    read_adata,
)

check_methods = ["Cellpose_1_DAPI_PolyT", "Baysor_2D_Cellpose_1_DAPI_PolyT_0.8", "vpt_2D_DAPI_PolyT", "Negative_Control_Voronoi"]
n_samples = 2

checks = []
for cohort in cohorts:
    for method in check_methods:
        adata = read_adata(cohort, method)
        if adata is None:
            continue
        adata = adata[adata.obs["sample"].isin(sorted(adata.obs["sample"].unique())[:n_samples])].copy()
        counts = adata.obs["n_counts"] if "n_counts" in adata.obs else pd.Series(np.asarray(adata.X.sum(1)).ravel(), index=adata.obs_names)
        adata.obs["depth"] = counts.groupby(adata.obs["sample"]).transform(lambda x: pd.qcut(x.rank(method="first"), 4, labels=False))
        adata.obs["low_vsi_cell"] = low_vsi_per_cell(adata, method, base).to_numpy() >= 0.5
        for (depth, low), obs in adata.obs.groupby(["depth", "low_vsi_cell"]):
            f1 = compute_marker_F1_score(adata[obs.index].copy(), celltype_name="cell_type_revised")
            f1["fpr"] = f1["fp"] / (len(obs) - f1["tp"] - f1["fn"]).where(lambda n: n > 0)
            checks.append(f1.assign(cohort=cohort, method=method, depth=depth, low_vsi_cell=low))
        del adata
checks = pd.concat(checks)

In [ ]:
strata = checks.groupby(["cohort", "method", "sample", "depth", "low_vsi_cell"])[["f1_score", "precision", "recall", "fpr"]].mean().unstack("low_vsi_cell")
diff = (strata.xs(True, axis=1, level=1) - strata.xs(False, axis=1, level=1)).dropna().reset_index()
cols = ["f1_score", "precision", "recall", "fpr"]
print(pd.concat({"mean diff": diff.groupby("cohort")[cols].mean(), "share > 0": (diff[cols] > 0).groupby(diff["cohort"]).mean()}, axis=1).round(3).to_string())

fig, axs = plt.subplots(1, 2, figsize=(9, 3.5))
for ax, col, label in [(axs[0], "f1_score", "Marker F1"), (axs[1], "fpr", "Marker false positive rate")]:
    sns.stripplot(diff, x=col, y="cohort", hue="method", dodge=True, size=4, ax=ax, legend=ax is axs[1])
    ax.axvline(0, color="k", lw=0.8)
    ax.set(xlabel=f"{label}\nlow-VSI cells minus rest", ylabel="")
axs[1].legend(bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False, fontsize=7)
plt.tight_layout()
plt.savefig(plot_path / "z_integrity_marker_check.png", dpi=200, bbox_inches="tight")
plt.show()

## 5. Low-VSI share per gene (segmentation free)

For each gene, the % of its transcripts on signal pixels that fall on low-VSI pixels. Genes are grouped by the cell type they mark (ABC Atlas positive markers, genes marking one type only). Highly expressed genes cluster in dense tissue, so the transcript count is shown beside the share.

In [ ]:
from cellseg_benchmark._constants import true_cluster, vascular_celltypes
from cellseg_benchmark.metrics import get_positive_markers, low_vsi_per_gene

genes = []
for cohort in cohorts:
    f = base / "metrics" / cohort / "z_integrity" / "low_vsi_per_gene.csv"
    if not f.exists():
        samples = sorted(df.loc[df["cohort"] == cohort, "sample"].unique())
        pd.concat(low_vsi_per_gene(s, base).assign(sample=s) for s in samples).to_csv(f, index=False)
    genes.append(pd.read_csv(f).assign(cohort=cohort))
genes = pd.concat(genes)
genes = genes[~genes["gene"].str.startswith("Blank")].groupby(["cohort", "gene"])[["n", "n_low"]].sum()
genes["low_pct"] = 100 * genes["n_low"] / genes["n"]

pos = get_positive_markers(subset_genes=set(genes.index.get_level_values("gene")))
owner = pd.DataFrame([(g, true_cluster.get(c, c)) for c, gs in pos.items() for g in gs], columns=["gene", "type"]).drop_duplicates()
owner = owner[~owner["gene"].duplicated(keep=False)].set_index("gene")["type"]
genes["type"] = genes.index.get_level_values("gene").map(owner)
genes["vascular"] = genes["type"].isin(vascular_celltypes)
genes.groupby(["cohort", "type"])["low_pct"].median().unstack(0).round(1).sort_values(cohorts[0], ascending=False)

In [ ]:
from scipy.stats import mannwhitneyu

fig, axs = plt.subplots(1, len(cohorts), figsize=(5 * len(cohorts), 4.5), sharey=False)
for ax, cohort in zip(axs, cohorts):
    g = genes.loc[cohort]
    ax.scatter(g["n"], g["low_pct"], s=6, c=np.where(g["vascular"], "#FF6464", "0.7"))
    p = mannwhitneyu(g.loc[g["vascular"], "low_pct"], g.loc[g["type"].notna() & ~g["vascular"], "low_pct"]).pvalue
    ax.set(xscale="log", xlabel="Transcripts on signal pixels", ylabel="Low-VSI transcripts (%)",
           title=f"{cohort}: vascular markers (red) vs other markers p={p:.1g}")
plt.tight_layout()
plt.savefig(plot_path / "z_integrity_per_gene.png", dpi=200, bbox_inches="tight")
plt.show()